<a href="https://colab.research.google.com/github/Lucianadeoliveira/gaTE-lab/blob/main/FRALCAT/ANALYSIS_AND_PROTEIN_INFERENCE/general_take_information_a3m_and_Uniprot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This script opens the .a3m file and takes the information about each entrance that satisfies the condition: seq. similarity>0.75. In this version it is taking all the information in the .a3m file.
Remember: this file is one of the outputs of alphaFold.
In the second part, it is taking the Uniprot codes that are present in the a3m file and is using it to acess the information in the Uniprot website and saving all the information about the protein, organism, gene, etc

**In the first part it is reading and saving the details in the a3m file**

In [ ]:
# import data from googleDrive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Verification and Generation of `details_data_a3m.csv`

If the file `details_data_a3m.csv` is **already present** in the subdirectories, this step can be **skipped** to save time, as it involves accessing the UniProt database and may be time-consuming.  
If the file is **not found**, the script will automatically locate `.a3m` files in each folder and generate the required `details_data_a3m.csv` for downstream analysis.


In [ ]:
import os
import re
import csv

# =========================================================
# User-defined input: define the root folder with outputs
# =========================================================
root_dir = '/content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/Dados/Dados/teste'

# =========================================================
# Filenames to check or generate (do not change)
# =========================================================
# These filenames are required for the workflow and should not be modified.
# They are automatically created or checked during processing.
file1 = 'details_data_a3m.csv'
file2 = 'information_proteins_uniprot.csv'

# =========================================================
# Main function to check for file presence and trigger processing
# =========================================================
def check_files_and_calculate(root_dir, file1, file2):
    file1_count = 0
    file2_count = 0
    missing_files_dirs = []

    for dirpath, dirnames, filenames in os.walk(root_dir):
        if os.path.basename(dirpath).endswith('_env'):
            continue

        print(f"Checking directory: {dirpath}")

        file1_present = file1 in filenames
        file2_present = file2 in filenames

        if file1_present:
            file1_count += 1
            print(f"Found {file1} in {dirpath}")
        if file2_present:
            file2_count += 1
            print(f"Found {file2} in {dirpath}")

        if not file1_present or not file2_present:
            missing_files_dirs.append(dirpath)

    if missing_files_dirs:
        print("One or both files are missing in some directories. Performing calculations...")
        for dirpath in missing_files_dirs:
            print(f"Performing calculations for directory: {dirpath}")
            your_calculation_function(dirpath)
    else:
        print(f"{file1} found in {file1_count} directories.")
        print(f"{file2} found in {file2_count} directories.")
        print("Both files are present in all necessary directories.")

# =========================================================
# Processing function: parse .a3m files and extract UniRef codes
# =========================================================
def your_calculation_function(dirpath):
    print(f"Calculating for directory: {dirpath}")

    a3m_file = next((f for f in os.listdir(dirpath) if f.endswith('.a3m')), None)

    if a3m_file:
        a3m_path = os.path.join(dirpath, a3m_file)
        print(f"Found .a3m file: {a3m_path}")

        with open(a3m_path, 'r') as file:
            content = file.read()

            codes = re.findall(r'>UniRef100_(.*?)\n', content)

            filtered_lines = []
            for line in codes:
                parts = line.split('\t')
                if len(parts) >= 3:
                    try:
                        identity_value = float(parts[2])
                        if identity_value >= 0.5:
                            filtered_lines.append(line)
                    except ValueError:
                        pass
                else:
                    # Add default 'not found' if information is missing
                    filtered_lines.append(f"{line}\tnot found\tnot found")

            csv_filename = os.path.join(dirpath, file1)
            with open(csv_filename, 'w', newline='') as csvfile:
                writer = csv.writer(csvfile)
                for line in filtered_lines:
                    writer.writerow(line.split('\t'))

            print(f"Filtered data saved to: {csv_filename}")
    else:
        print("No .a3m file found in this directory")

# =========================================================
# Run the pipeline
# =========================================================
check_files_and_calculate(root_dir, file1, file2)


Checking directory: /content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/Dados/Dados/teste
Checking directory: /content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/Dados/Dados/teste/teste1
Found details_data_a3m.csv in /content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/Dados/Dados/teste/teste1
Checking directory: /content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/Dados/Dados/teste/teste2
Found details_data_a3m.csv in /content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/Dados/Dados/teste/teste2
One or both files are missing in some directories. Performing calculations...
Performing calculations for directory: /content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/Dados/Dados/teste
Calculating for directory: /content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/Dados/Dados/teste
No .a3m file found in this directory
Performing calculations for directory: /content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/Dados/Dados/teste/teste1
Calculating for directory: /content/drive/MyDrive

UniProt Data Retrieval
This section reads UniProt IDs from the details_data_a3m.csv file and automatically fetches detailed protein information from the UniProt database. The retrieved data is saved as information_proteins_uniprot.csv in each respective directory for further analysis.

In [ ]:
import os
import re
import csv
import requests

def process_all(root_dir):
    """
    Iterates through directories within the specified root directory, generating
    or updating relevant annotation files based on AlphaFold alignment outputs.
    """
    for dirpath, _, filenames in os.walk(root_dir):
        if os.path.basename(dirpath).endswith('_env'):
            continue

        print(f"\nProcessing directory: {dirpath}")

        details_path = os.path.join(dirpath, 'details_data_a3m.csv')
        uniprot_path = os.path.join(dirpath, 'information_proteins_uniprot.csv')

        # Step 1: Generate 'details_data_a3m.csv' if not already present
        if not os.path.exists(details_path):
            generate_details_data_a3m(dirpath)
        else:
            print("'details_data_a3m.csv' already exists.")

        # Step 2: Generate 'information_proteins_uniprot.csv' with subcellular annotations
        if not os.path.exists(uniprot_path):
            fetch_uniprot_data(dirpath)
        else:
            print("'information_proteins_uniprot.csv' already exists.")

def generate_details_data_a3m(dirpath):
    """
    Extracts UniRef100 identifiers from A3M alignment files and filters them
    based on sequence identity (≥ 0.5). The results are saved to a CSV file.
    """
    print("Generating 'details_data_a3m.csv'...")
    a3m_file = None
    for filename in os.listdir(dirpath):
        if filename.endswith('.a3m'):
            a3m_file = filename
            break

    if not a3m_file:
        print("No A3M alignment file was found in the directory.")
        return

    a3m_path = os.path.join(dirpath, a3m_file)
    with open(a3m_path, 'r') as file:
        content = file.read()

    codes = re.findall(r'>UniRef100_(.*?)\n', content)
    filtered_lines = []

    for line in codes:
        parts = line.split('\t')
        if len(parts) >= 3:
            try:
                identity = float(parts[2])
                if identity >= 0.5:
                    filtered_lines.append(line)
            except ValueError:
                continue

    csv_path = os.path.join(dirpath, 'details_data_a3m.csv')
    with open(csv_path, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        for line in filtered_lines:
            writer.writerow(line.split('\t'))

    print(f"File saved: {csv_path}")

def fetch_uniprot_data(dirpath):
    """
    For each UniProt ID extracted from the alignment, this function retrieves
    biological metadata from UniProt and appends subcellular location information.
    The final dataset is saved as a tab-delimited file.
    """
    print("Retrieving UniProt metadata including subcellular localization...")

    csv_filename = os.path.join(dirpath, 'details_data_a3m.csv')
    if not os.path.exists(csv_filename):
        print("'details_data_a3m.csv' not found. Skipping UniProt metadata retrieval.")
        return

    with open(csv_filename, 'r') as csvfile:
        reader = csv.reader(csvfile)
        uniprot_ids = [row[0] for row in reader if row]

    all_data = []
    header = [
        "Entry", "Entry Name", "Reviewed", "Protein names",
        "Gene Names", "Organism", "Length", "Subcellular Location"
    ]
    all_data.append(header)

    for uniprot_id in uniprot_ids:
        tsv_data = get_uniprot_data(uniprot_id)
        txt_data = fetch_uniprot_file(uniprot_id)
        subcell_location = extract_subcellular_location(txt_data) if txt_data else ""

        if tsv_data and len(tsv_data) > 1:
            row = tsv_data[1]
            row_extended = row[:7] + [subcell_location]
            all_data.append(row_extended)
        else:
            all_data.append([uniprot_id] + ['not found'] * 6 + [subcell_location])

    output_path = os.path.join(dirpath, 'information_proteins_uniprot.csv')
    with open(output_path, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile, delimiter='\t')
        writer.writerows(all_data)

    print(f"File saved: {output_path}")

def get_uniprot_data(uniprot_id):
    """
    Retrieves tabular metadata for a given UniProt ID from the UniProt website.
    """
    url = f'https://www.uniprot.org/uniprot/{uniprot_id}.tsv'
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            lines = response.text.splitlines()
            return [line.split('\t') for line in lines]
        else:
            return None
    except Exception:
        return None

def fetch_uniprot_file(entry_id):
    """
    Retrieves the full UniProt flat file in text format for a given entry.
    """
    url = f"https://rest.uniprot.org/uniprotkb/{entry_id}.txt"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            return response.text
        else:
            return None
    except Exception:
        return None

def extract_subcellular_location(text):
    """
    Parses UniProt flat file content to extract subcellular location annotation, if present.
    """
    if not text:
        return ""
    match = re.search(r"CC   -!- SUBCELLULAR LOCATION:\s*([^\{]+)", text)
    if match:
        return match.group(1).strip()
    return ""

# Execute the processing pipeline using the 'root_dir' variable defined earlier in the notebook
process_all(root_dir)



Processing directory: /content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/Dados/Dados/teste
Generating 'details_data_a3m.csv'...
No A3M alignment file was found in the directory.
Retrieving UniProt metadata including subcellular localization...
'details_data_a3m.csv' not found. Skipping UniProt metadata retrieval.

Processing directory: /content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/Dados/Dados/teste/teste1
'details_data_a3m.csv' already exists.
Retrieving UniProt metadata including subcellular localization...
File saved: /content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/Dados/Dados/teste/teste1/information_proteins_uniprot.csv

Processing directory: /content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/Dados/Dados/teste/teste2
'details_data_a3m.csv' already exists.
Retrieving UniProt metadata including subcellular localization...
File saved: /content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/Dados/Dados/teste/teste2/information_proteins_uniprot.csv


UniProt Organism Data Retrieval
This section scans all subdirectories starting from the root directory to extract UniProt IDs from local CSV files (details_data_a3m.csv). It then fetches the corresponding organism information from the UniProt database. The collected organism names are appended to a consolidated CSV file for further analysis.

**Important**: Remember to update the output file path variable to the desired location where you want the organism list to be saved.

In [ ]:
import os
import csv
import requests

# Root directory containing subdirectories with 'details_data_a3m.csv' files
root_dir = '/content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/Dados/Dados/Dados-Dora'

def fetch_uniprot_data(dirpath, output_csv):
    """
    Reads UniProt IDs from 'details_data_a3m.csv' in the specified directory,
    queries the UniProt database for the corresponding organism names,
    and appends these organisms to a consolidated CSV file.
    """
    print(f"Fetching UniProt data for directory: {dirpath}...")

    csv_filename = os.path.join(dirpath, 'details_data_a3m.csv')
    if not os.path.exists(csv_filename):
        print(f"{csv_filename} does not exist.")
        return

    organisms = []
    with open(csv_filename, 'r') as csvfile:
        reader = csv.reader(csvfile)
        uniprot_ids = [row[0] for row in reader]  # Assumes UniProt IDs are in the first column

    for uniprot_id in uniprot_ids:
        organism = get_organism_from_uniprot(uniprot_id)
        if organism:
            organisms.append(organism)

    with open(output_csv, 'a', newline='') as csvfile:
        writer = csv.writer(csvfile)
        for organism in organisms:
            writer.writerow([organism])

    print(f"Organism data appended to: {output_csv}")

def get_organism_from_uniprot(uniprot_id):
    """
    Queries the UniProt database for the organism name associated with the given UniProt ID.
    Returns the organism name as a string if successful; otherwise, returns None.
    """
    url = f'https://www.uniprot.org/uniprot/{uniprot_id}.tsv'
    response = requests.get(url)

    if response.status_code == 200:
        lines = response.text.splitlines()
        reader = csv.DictReader(lines, delimiter='\t')
        for row in reader:
            return row.get('Organism')
    else:
        print(f"Failed to fetch data for UniProt ID: {uniprot_id}")
    return None

def check_and_fetch_uniprot_data(output_csv):
    """
    Traverses all subdirectories within the globally defined 'root_dir' and,
    for each, invokes fetch_uniprot_data to retrieve and append organism data.
    Ensures the output CSV file includes a header and is created if missing.
    """
    if not os.path.exists(output_csv):
        with open(output_csv, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['Organism'])  # Write header row

    for dirpath, _, _ in os.walk(root_dir):
        fetch_uniprot_data(dirpath, output_csv)

# Output CSV path to save all organism data
output_csv = '/content/drive/MyDrive/GateLab-ProjetoTT5- Marie Anne/resultados/25-10-24-nova-montagem/all_organisms.csv'

# Execute the function
check_and_fetch_uniprot_data(output_csv)
